# Stakeholder Gym — DPO training (Kaggle T4 x2)

Direct Preference Optimization on Qwen 2.5-3B. Bypasses GRPO plateau by training on synthesized (prompt, chosen, rejected) pairs from our scripted scenarios. **No rollouts during training.** ~25-40 min total.

**Settings → Accelerator → GPU T4 x2** before running.

Run cells top-to-bottom. Don't skip.

In [ ]:
# Cell 1 — GPU check
!nvidia-smi
import torch
n = torch.cuda.device_count()
assert n >= 2, f'Need 2 GPUs, got {n}. Settings -> Accelerator -> GPU T4 x2.'
for i in range(n):
    print(f'[{i}] {torch.cuda.get_device_name(i)}  {torch.cuda.get_device_properties(i).total_memory/1e9:.1f}GB')

In [ ]:
# Cell 2 — install deps
!pip install -q -U transformers accelerate bitsandbytes peft trl datasets pydantic networkx pyyaml matplotlib

In [ ]:
# Cell 3 — fresh clone (delete any stale copy first; safe to rerun)
%cd /kaggle/working
!rm -rf /kaggle/working/meta
!git clone --depth 1 https://github.com/SAISriram19/meta.git /kaggle/working/meta
%cd /kaggle/working/meta
!git log --oneline -1
# Verify the DPO files are in the cloned repo.
!ls scripts/build_dpo_pairs.py scripts/train_dpo_ddp.py 2>&1
!grep -n 'class DPOTrainer\|DPOConfig' scripts/train_dpo_ddp.py | head -5

In [ ]:
# Cell 4 — accelerate config (2 GPUs, bf16)
import os, yaml
cfg = {
    'compute_environment': 'LOCAL_MACHINE',
    'distributed_type': 'MULTI_GPU',
    'mixed_precision': 'bf16',
    'num_processes': 2,
    'num_machines': 1,
    'machine_rank': 0,
    'main_training_function': 'main',
    'gpu_ids': 'all',
    'rdzv_backend': 'static',
    'same_network': True,
    'tpu_use_cluster': False,
    'tpu_use_sudo': False,
    'use_cpu': False,
}
os.makedirs('/root/.cache/huggingface/accelerate', exist_ok=True)
with open('/root/.cache/huggingface/accelerate/default_config.yaml', 'w') as f:
    yaml.dump(cfg, f)
print('accelerate config written')

In [ ]:
# Cell 5 — build DPO preference pairs from scenarios. ~5 sec.
!python scripts/build_dpo_pairs.py --out data/dpo_pairs.jsonl --cap-per-scenario 80
!wc -l data/dpo_pairs.jsonl

In [ ]:
# Cell 6 — BEFORE numbers (rule-based baselines, ~90s)
!python scripts/run_eval.py \
  --policies sycophant,keyword_principled,memory_aware \
  --scenarios L0_launch,L2_strategic_shift \
  --seeds 0,1,2 \
  --out eval_outputs/pre_train_dpo

In [ ]:
# Cell 7 — DUAL-GPU DPO TRAINING. ~25-35 min.
# Batch math: per_device(2) * num_gpus(2) * grad_accum(4) = 16 effective.
# 179 pairs / 16 = ~11 steps per epoch, 2 epochs = ~22 steps.
!accelerate launch --num_processes=2 --mixed_precision=bf16 \
    scripts/train_dpo_ddp.py \
    --model Qwen/Qwen2.5-3B-Instruct \
    --data data/dpo_pairs.jsonl \
    --epochs 3 \
    --lr 5e-6 \
    --per-device-batch 2 \
    --grad-accum 4 \
    --beta 0.1 \
    --output /kaggle/working/outputs/dpo-stakeholder

In [ ]:
# Cell 8 — plot DPO training curves (loss + reward margins)
import json
import matplotlib.pyplot as plt

with open('/kaggle/working/outputs/dpo-stakeholder-lora/log.json') as f:
    log = json.load(f)
steps = [h['step'] for h in log if 'loss' in h]
loss = [h['loss'] for h in log if 'loss' in h]
margins = [h.get('rewards/margins', None) for h in log if 'loss' in h]
acc = [h.get('rewards/accuracies', None) for h in log if 'loss' in h]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(steps, loss, marker='o', alpha=0.6, color='C0')
axes[0].set_title('DPO loss')
axes[0].set_xlabel('step')
axes[0].grid(alpha=0.3)

if any(m is not None for m in margins):
    axes[1].plot(steps, [m or 0 for m in margins], marker='o', alpha=0.6, color='C1')
    axes[1].set_title('reward margin (chosen − rejected)')
    axes[1].set_xlabel('step')
    axes[1].grid(alpha=0.3)
    axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)

if any(a is not None for a in acc):
    axes[2].plot(steps, [a or 0 for a in acc], marker='o', alpha=0.6, color='C2')
    axes[2].set_title('preference accuracy')
    axes[2].set_xlabel('step')
    axes[2].grid(alpha=0.3)
    axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('/kaggle/working/outputs/dpo_training_curves.png', dpi=120)
plt.show()
print(f'loss first={loss[0]:.4f} last={loss[-1]:.4f}')
if any(m is not None for m in margins):
    last_m = next((m for m in reversed(margins) if m is not None), None)
    print(f'final reward margin: {last_m:+.4f}')
if any(a is not None for a in acc):
    last_a = next((a for a in reversed(acc) if a is not None), None)
    print(f'final preference accuracy: {last_a:.3f}')

In [ ]:
# Cell 9 — post-train eval. Load LoRA, run on L0+L2, 3 seeds.
import sys, json, torch
from pathlib import Path
sys.path.insert(0, '.')
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from env.environment import StakeholderEnv
from eval.harness import EvalConfig, run_eval
from eval.policies import LLM_SYSTEM_PROMPT
from scripts.train import format_prompt, parse_completion

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
LORA_PATH = '/kaggle/working/outputs/dpo-stakeholder-lora'

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type='nf4')
tokenizer = AutoTokenizer.from_pretrained(LORA_PATH)
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb, torch_dtype=torch.bfloat16, device_map='auto', trust_remote_code=True)
model = PeftModel.from_pretrained(base, LORA_PATH)
model.eval()

def make_trained_policy():
    def act(ctx):
        obs_json = format_prompt(ctx.observation, ctx.env)
        prompt = LLM_SYSTEM_PROMPT + '\n\nOBSERVATION:\n' + obs_json + '\n\nReturn ONE action as strict JSON.'
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1800).to(model.device)
        out = model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.4, pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        return parse_completion(text, ctx.env)
    return act

cfg_eval = EvalConfig(
    policies={'dpo_qwen3b': make_trained_policy()},
    scenarios=['L0_launch', 'L2_strategic_shift'],
    seeds=[0, 1, 2],
    out_dir=Path('eval_outputs/post_train_dpo'),
)
run_eval(cfg_eval, verbose=True)

In [ ]:
# Cell 10 — before/after comparison
import json
with open('eval_outputs/pre_train_dpo/summary.json') as f:
    pre = json.load(f)
with open('eval_outputs/post_train_dpo/summary.json') as f:
    post = json.load(f)
pre_rows = pre['cells'] if isinstance(pre, dict) and 'cells' in pre else pre
post_rows = post['cells'] if isinstance(post, dict) and 'cells' in post else post

print(f'{"kind":<7} {"policy":<22} {"scenario":<22} {"reward":>8} {"bad":>5} {"terminal":>8} {"princ":>6}')
print('-' * 92)
for r in pre_rows:
    print(f'BEFORE  {r["policy"]:<22} {r["scenario"]:<22} {r["total_reward_mean"]:>+8.3f} {r["bad_agreements_mean"]:>5.1f} {r["terminal_score_mean"]:>+8.3f} {r["principled_mean"]:>6.1f}')
print()
for r in post_rows:
    print(f'AFTER   {r["policy"]:<22} {r["scenario"]:<22} {r["total_reward_mean"]:>+8.3f} {r["bad_agreements_mean"]:>5.1f} {r["terminal_score_mean"]:>+8.3f} {r["principled_mean"]:>6.1f}')

In [ ]:
# Cell 11 — package artifacts for download
import shutil, tarfile, os
out = '/kaggle/working/outputs'
shutil.copy('eval_outputs/pre_train_dpo/summary.json', f'{out}/pre_train_dpo_summary.json')
shutil.copy('eval_outputs/post_train_dpo/summary.json', f'{out}/post_train_dpo_summary.json')
with tarfile.open(f'{out}/dpo-stakeholder-lora.tar.gz', 'w:gz') as t:
    t.add(f'{out}/dpo-stakeholder-lora', arcname='dpo-stakeholder-lora')
for f in sorted(os.listdir(out)):
    p = f'{out}/{f}'
    sz = os.path.getsize(p) if os.path.isfile(p) else '-'
    print(f'  {f}  {sz}')